# Wikipedia web scraping — Falcon 9 launches
Retrieve public launch tables with `pandas.read_html`. Page structure may change; keep the raw table and validate columns before use.

In [ ]:
import pandas as pd, requests, io
from pathlib import Path
DATA=Path('../data');DATA.mkdir(exist_ok=True)
url='https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches'
html=requests.get(url,headers={'User-Agent':'Mozilla/5.0 (educational research)'},timeout=45)
html.raise_for_status()
tables=pd.read_html(io.StringIO(html.text))
print('HTML tables:',len(tables))
for i,t in enumerate(tables[:15]): print(i,t.shape,str(t.columns.tolist())[:160])

In [ ]:
# Keep all tables containing a flight number; preserve source columns.
selected=[]
for i,t in enumerate(tables):
    cols=' '.join(map(str,t.columns)).lower()
    if 'flight' in cols or 'no.' in cols:
        t=t.copy();t['SourceTable']=i;selected.append(t)
print('Candidate tables:',len(selected))
if selected:
    raw=pd.concat(selected,ignore_index=True,sort=False)
    raw.to_csv(DATA/'wikipedia_launch_tables_raw.csv',index=False)
    display(raw.head())
else: print('Page layout changed: inspect the table list above and select relevant tables manually.')